In [1]:
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
uploaded = files.upload()

Saving Online Retail.xlsx to Online Retail.xlsx


In [3]:
df = pd.read_excel("Online Retail.xlsx")
df = df.drop_duplicates().copy()

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

data = df[
    (df["UnitPrice"] > 0) &
    (df["Quantity"] > 0) &
    (df["InvoiceDate"].notna()) &
    (df["StockCode"].notna())
].copy()

data["StockCode"] = data["StockCode"].astype(str)
data["Month"] = data["InvoiceDate"].dt.month
data["Hour"] = data["InvoiceDate"].dt.hour
data["DayOfWeek"] = data["InvoiceDate"].dt.dayofweek

# Reproducible sample for faster evaluation
sample = data[
    ["UnitPrice", "Quantity", "StockCode",
     "Month", "Hour", "DayOfWeek"]
].sample(
    n=min(20000, len(data)),
    random_state=42
).reset_index(drop=True)

print("Rows evaluated:", len(sample))
print("Unique products:", sample["StockCode"].nunique())

Rows evaluated: 20000
Unique products: 2814


In [5]:
def estimate_dml(sample, controls):
    T = sample["UnitPrice"].to_numpy()
    Y = sample["Quantity"].to_numpy()
    X = sample[controls].copy()

    kf = KFold(n_splits=2, shuffle=True, random_state=42)

    t_res = np.zeros(len(sample))
    y_res = np.zeros(len(sample))

    for train_idx, test_idx in kf.split(X):
        transformers = []

        if "StockCode" in controls:
            transformers.append((
                "product",
                OneHotEncoder(handle_unknown="ignore"),
                ["StockCode"]
            ))

        time_cols = [c for c in controls if c != "StockCode"]
        if time_cols:
            transformers.append(("time", "passthrough", time_cols))

        prep_t = ColumnTransformer(transformers)
        prep_y = ColumnTransformer(transformers)

        model_t = make_pipeline(prep_t, Ridge(alpha=10.0))
        model_y = make_pipeline(prep_y, Ridge(alpha=10.0))

        model_t.fit(X.iloc[train_idx], T[train_idx])
        model_y.fit(X.iloc[train_idx], Y[train_idx])

        t_res[test_idx] = T[test_idx] - model_t.predict(X.iloc[test_idx])
        y_res[test_idx] = Y[test_idx] - model_y.predict(X.iloc[test_idx])

    denom = np.mean(t_res ** 2)
    if denom <= 1e-10:
        return np.nan, denom

    theta = np.mean(t_res * y_res) / denom
    return theta, denom

In [6]:
specifications = {
    "Time controls only": ["Month", "Hour", "DayOfWeek"],
    "Product and time controls": [
        "StockCode", "Month", "Hour", "DayOfWeek"
    ]
}

results = []

for name, controls in specifications.items():
    theta, residual_variation = estimate_dml(sample, controls)

    results.append({
        "Specification": name,
        "SampleSize": len(sample),
        "Controls": ", ".join(controls),
        "PreliminaryCoefficient": theta,
        "ResidualPriceVariation": residual_variation
    })

evaluation = pd.DataFrame(results)
display(evaluation)

,Specification,SampleSize,Controls,PreliminaryCoefficient,ResidualPriceVariation
0,Time controls only,20000,"Month, Hour, DayOfWeek",-0.005836,3944.112720
1,Product and time controls,20000,"StockCode, Month, Hour, DayOfWeek",-0.002905,3877.818258


In [7]:
evaluation.to_csv(
    "day15_dml_evaluation_summary.csv",
    index=False
)

print("Day 15 evaluation report saved!")

Day 15 evaluation report saved!


In [8]:
files.download("day15_dml_evaluation_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>